# LSTM Fake Review Detection
**Model:** Bidirectional LSTM

**Dataset:** `fake reviews dataset.csv` (Kaggle)

**Labels:** CG (Computer Generated) = 1 (fake) | OR (Original) = 0 (genuine)

**Training sample:** All 40,432 rows (20,216 fake + 20,216 genuine) — full dataset, no sampling needed.

---
### Instructions
1. Go to **Runtime → Change runtime type → T4 GPU** before running.
2. Upload `fake reviews dataset.csv` to your Google Drive.
3. Run all cells in order.
4. Download `fake_lstm.keras` and `fake_lstm_tokenizer.json` and place them in your `data/` folder.

In [ ]:
import torch
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── UPDATE THIS PATH if the file is inside a folder ──
CSV_PATH = '/content/drive/MyDrive/fake reviews dataset.csv'

In [ ]:
import csv
import random
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import json

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────
MAX_LEN    = 100      # max tokens per review
VOCAB_SIZE = 30_000   # top N words kept
EMBED_DIM  = 128
LSTM_UNITS = 64
BATCH_SIZE = 256
EPOCHS     = 10
# ─────────────────────────────────────────────────────────────────

In [ ]:
# Load fake reviews dataset
# Labels: CG (Computer Generated) = 1 (fake), OR (Original) = 0 (genuine)
print('Loading data ...')
df = pd.read_csv(CSV_PATH)
print(f'Total rows: {len(df):,}')
print(df['label'].value_counts())

df['label_int'] = df['label'].map({'CG': 1, 'OR': 0})
df = df.dropna(subset=['text_', 'label_int'])
df['text_'] = df['text_'].astype(str)

# Shuffle
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f'\nUsing {len(df):,} rows  ({df["label_int"].sum():,} fake / {(df["label_int"]==0).sum():,} genuine)')

In [ ]:
texts  = df['text_'].tolist()
labels = df['label_int'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=SEED, stratify=labels
)
print(f'Train: {len(X_train):,}   Test: {len(X_test):,}')

In [ ]:
# Tokenise
print('Fitting tokenizer ...')
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)
print(f'Vocabulary size: {len(tokenizer.word_index):,}')

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, truncating='post', padding='post')
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test),  maxlen=MAX_LEN, truncating='post', padding='post')

y_train_arr = np.array(y_train)
y_test_arr  = np.array(y_test)
print(f'X_train shape: {X_train_seq.shape}')

In [ ]:
# Build Bidirectional LSTM
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM, input_length=MAX_LEN),
    Bidirectional(LSTM(LSTM_UNITS, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(LSTM_UNITS // 2)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=2, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, verbose=1)
]

history = model.fit(
    X_train_seq, y_train_arr,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks
)

In [ ]:
# Evaluate
loss, accuracy = model.evaluate(X_test_seq, y_test_arr, batch_size=BATCH_SIZE, verbose=0)
print(f'Test Loss     : {loss:.4f}')
print(f'Test Accuracy : {accuracy:.1%}')

y_pred = (model.predict(X_test_seq, batch_size=BATCH_SIZE) > 0.5).astype(int).flatten()
print()
print(classification_report(y_test_arr, y_pred, target_names=['Genuine', 'Fake']))

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Save model and tokenizer
model.save('fake_lstm.keras')
print('Model saved: fake_lstm.keras')

with open('fake_lstm_tokenizer.json', 'w', encoding='utf-8') as f:
    f.write(tokenizer.to_json())
print('Tokenizer saved: fake_lstm_tokenizer.json')

In [ ]:
# Download both files
from google.colab import files
files.download('fake_lstm.keras')
files.download('fake_lstm_tokenizer.json')

## Next steps
Place both files in your project's `data/` folder:
```
data/fake_lstm.keras
data/fake_lstm_tokenizer.json
```